In [ ]:
# Cell 1: Import all necessary libraries
from __future__ import absolute_import, division, print_function, unicode_literals

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau  # CHANGE: Added callbacks for better training
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns  # CHANGE: Added seaborn for better confusion matrix visualization



In [ ]:
# Cell 2: Load and preprocess the MNIST dataset
# Load the MNIST dataset containing handwritten digits (0-9)
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Print original data shapes for verification
print(f"Original training data shape: {x_train.shape}")
print(f"Original test data shape: {x_test.shape}")

# Normalize pixel values to range [0,1] for better training stability
x_train, x_test = x_train / 255.0, x_test / 255.0

In [ ]:
# Cell 3: CHANGE - Create validation split from training data
# Create a validation set from training data for better model monitoring
validation_split = 0.1  # Use 10% of training data for validation
val_size = int(len(x_train) * validation_split)
x_val = x_train[-val_size:]
y_val = y_train[-val_size:]
x_train = x_train[:-val_size]
y_train = y_train[:-val_size]

# Print dataset sizes after split
print(f"Training set size: {x_train.shape[0]}")
print(f"Validation set size: {x_val.shape[0]}")
print(f"Test set size: {x_test.shape[0]}")

In [ ]:
# Cell 4: Visualize sample training images
# Define class names for digit labels
class_names = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

# Create a 5x5 grid showing 25 sample images from training set
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)  # Create 5x5 grid of subplots
    plt.xticks([])  # Remove x-axis ticks
    plt.yticks([])  # Remove y-axis ticks
    plt.grid(False)  # Disable grid
    plt.imshow(x_train[i], cmap='gray')  # CHANGE: Added grayscale colormap for better visualization
    plt.xlabel(class_names[y_train[i]])  # Label each image with its digit
plt.suptitle("Sample Training Images", fontsize=16)  # CHANGE: Added title
plt.show()


In [ ]:
# Cell 5: Reshape data for CNN input
# Reshape data to add channel dimension for CNN input (28x28x1)
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_val = x_val.reshape(x_val.shape[0], 28, 28, 1)  # CHANGE: Reshape validation set too
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)

# Verify shapes after reshaping
print(f"Reshaped training data: {x_train.shape}")
print(f"Reshaped validation data: {x_val.shape}")
print(f"Reshaped test data: {x_test.shape}")

In [ ]:
# Cell 6: CHANGE - Build improved CNN architecture with regularization
# Build an enhanced CNN model with better regularization techniques
model = models.Sequential([
    # First convolutional block
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.BatchNormalization(),  # CHANGE: Added batch normalization for training stability
    layers.Conv2D(32, (3, 3), activation='relu'),  # CHANGE: Added second conv layer before pooling
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),  # CHANGE: Added dropout for regularization

    # Second convolutional block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),  # CHANGE: Added batch normalization
    layers.Conv2D(64, (3, 3), activation='relu'),  # CHANGE: Added second conv layer
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),  # CHANGE: Added dropout

    # Third convolutional block for deeper feature extraction
    layers.Conv2D(128, (3, 3), activation='relu'),  # CHANGE: Added third conv block
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    # Flatten and dense layers
    layers.Flatten(),
    layers.Dense(128, activation='relu'),  # CHANGE: Increased neurons from 64 to 128
    layers.BatchNormalization(),  # CHANGE: Added batch normalization
    layers.Dropout(0.5),  # CHANGE: Added dropout before final layer
    layers.Dense(10, activation='softmax')  # Output layer for 10 classes
])

In [ ]:
# Cell 7: Display model architecture
# Show detailed model architecture summary
print("=== Model Architecture ===")
model.summary()

In [ ]:
# Cell 8: CHANGE - Compile model with improved configuration
# Compile model with optimizer, loss function, and metrics
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # CHANGE: Simplified loss function (removed from_logits=True)
    metrics=['accuracy']
)

In [ ]:
# Cell 9: CHANGE - Define training callbacks for better control
# Define callbacks for enhanced training control
callbacks = [
    # Stop training early if validation loss doesn't improve for 3 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate when validation loss plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=0.0001,
        verbose=1
    )
]

In [ ]:
# Cell 10: CHANGE - Train model with validation data and callbacks
# Train the model with enhanced configuration
print("=== Training Model ===")
history = model.fit(
    x_train, y_train,
    batch_size=32,  # CHANGE: Added explicit batch size
    epochs=15,  # CHANGE: Increased epochs since we have early stopping
    validation_data=(x_val, y_val),  # CHANGE: Use proper validation set instead of test set
    callbacks=callbacks,  # CHANGE: Added callbacks for training control
    verbose=1
)

In [ ]:
# Cell 11: CHANGE - Enhanced visualization of training history
# Create comprehensive training history visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot training and validation accuracy over epochs
ax1.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
ax1.set_title('Model Accuracy Over Time')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_ylim([0.5, 1])
ax1.legend()
ax1.grid(True, alpha=0.3)  # CHANGE: Added grid for better readability

# Plot training and validation loss over epochs
ax2.plot(history.history['loss'], label='Training Loss', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
ax2.set_title('Model Loss Over Time')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)  # CHANGE: Added grid

plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Evaluate model on test set
# Evaluate final model performance on unseen test data
print("=== Model Evaluation ===")
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


In [ ]:
# Cell 13: CHANGE - Enhanced single prediction visualization with GPU
# Make predictions and visualize a sample with confidence scores
with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
    predictions = model.predict(x_test, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)

# Display sample prediction with confidence visualization
sample_idx = 1234
plt.figure(figsize=(8, 4))

# Show original image
plt.subplot(1, 2, 1)
plt.imshow(x_test[sample_idx].reshape(28, 28), cmap='gray')
plt.title(f'Test Image #{sample_idx}\nTrue Label: {y_test[sample_idx]}')
plt.axis('off')

# Show prediction confidence for all classes
plt.subplot(1, 2, 2)
plt.bar(class_names, predictions[sample_idx])
plt.title(f'Prediction Confidence\nPredicted: {predicted_classes[sample_idx]}')
plt.xlabel('Digit Class')
plt.ylabel('Probability')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
# Cell 14: Generate all predictions for evaluation
# Convert all predictions to class labels for comprehensive evaluation
print(f"Sample prediction probabilities: {predictions[1234]}")
print(f"Predicted class for sample {sample_idx}: {np.argmax(predictions[1234])}")
print(f"True class for sample {sample_idx}: {y_test[sample_idx]}")

In [ ]:
# Cell 15: CHANGE - Enhanced confusion matrix visualization
# Create and display a detailed confusion matrix using seaborn
print("=== Detailed Performance Analysis ===")
cm = confusion_matrix(y_test, predicted_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Enhanced CNN Digit Classification')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# Cell 16: Generate classification report
# Display detailed per-class performance metrics
print("=== Classification Report ===")
print(classification_report(y_test, predicted_classes, target_names=class_names))

In [ ]:
# Cell 17: CHANGE - Calculate comprehensive performance metrics
# Calculate and display all important performance metrics
print("=== Performance Metrics ===")
accuracy = sklearn.metrics.accuracy_score(y_test, predicted_classes)
f1_score = sklearn.metrics.f1_score(y_test, predicted_classes, average='weighted')
recall = sklearn.metrics.recall_score(y_test, predicted_classes, average='weighted')
precision = sklearn.metrics.precision_score(y_test, predicted_classes, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1_score:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")

In [ ]:
# Cell 18: CHANGE - Per-class accuracy analysis
# Analyze accuracy for each individual digit class
print("=== Per-Class Accuracy Analysis ===")
for i in range(10):
    # Create mask for current digit class
    class_mask = (y_test == i)
    # Calculate accuracy only for this class
    class_accuracy = accuracy_score(y_test[class_mask], predicted_classes[class_mask])
    class_count = np.sum(class_mask)
    print(f"Digit {i}: {class_accuracy:.4f} (n={class_count})")

In [ ]:
# Cell 19: CHANGE - Model confidence analysis
# Analyze the distribution of model confidence scores
plt.figure(figsize=(10, 6))
max_confidences = np.max(predictions, axis=1)

# Create histogram of confidence scores
plt.hist(max_confidences, bins=50, alpha=0.7, edgecolor='black')
plt.title('Distribution of Model Confidence Scores')
plt.xlabel('Maximum Probability')
plt.ylabel('Frequency')

# Add mean confidence line
plt.axvline(np.mean(max_confidences), color='red', linestyle='--',
           label=f'Mean: {np.mean(max_confidences):.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Display confidence statistics
print(f"Mean prediction confidence: {np.mean(max_confidences):.4f}")
print(f"Predictions with >90% confidence: {np.sum(max_confidences > 0.9) / len(max_confidences) * 100:.1f}%")
print(f"Predictions with >95% confidence: {np.sum(max_confidences > 0.95) / len(max_confidences) * 100:.1f}%")